# CLIP-Guided StyleGAN2 with 3 Learned Mapping Networks

## Overview

This notebook implements a pipeline where **3 separate mapping networks** (initialised from the pretrained StyleGAN2 mapping network) learn to convert CLIP image embeddings into the W+ latent space of a **StyleGAN2 synthesis network** to generate **256×256** face images.

### Architecture

```
Real Image ──► CLIP ViT-B/32 ──► 512-dim embedding ──► 3 Mapping Networks ──► W+ (14 × 512) ──► Synthesis Network ──► Generated Image
                                                                                                                           │
                                                                   ┌───────────────────────────────────────────────────────┘
                                                                   │
                                                                   ├──► Discriminator ──► Adversarial Loss
                                                                   │
                                                                   └──► CLIP ──► Cosine Similarity with Real ──► CLIP Loss
```

### Mapper-to-Layer Assignments

For 256×256 resolution, StyleGAN2 uses **14 style layers** (num_ws=14). Each mapper produces a single 512-dim **w** vector that is broadcast to its assigned layers:

| Mapper | Resolution Blocks | W Layers | # of w values |
|--------|------------------|----------|---------------|
| Mapper 0 | 4×4, 8×8 | 0, 1, 2, 3 | 4 × 512 |
| Mapper 1 | 16×16, 32×32 | 4, 5, 6, 7 | 4 × 512 |
| Mapper 2 | 64×64, 128×128, 256×256 | 8–13 | 6 × 512 |

**Total: 14 × 512 = 7,168 w values** forming the complete W+ tensor.

### Loss Function

$$\mathcal{L} = \mathcal{L}_{adv} + \lambda \cdot \mathcal{L}_{CLIP}$$

- $\mathcal{L}_{adv}$: Non-saturating generator loss from frozen Discriminator
- $\mathcal{L}_{CLIP} = 1 - \cos(\text{CLIP}(x_{real}),\; \text{CLIP}(x_{fake}))$
- $\lambda$: Adjustable weight (default 1.0)

### What is Frozen vs Trainable

| Component | Frozen? | Trainable? |
|-----------|---------|------------|
| CLIP ViT-B/32 | ✓ | ✗ |
| StyleGAN2 Synthesis Network | ✓ | ✗ |
| StyleGAN2 Discriminator | ✓ | ✗ |
| 3 Mapping Networks | ✗ | ✓ |

In [ ]:
!pip install torch torchvision ftfy regex tqdm
!pip install ninja matplotlib pillow

## Step 1 – Install Dependencies & Clone StyleGAN2 Repository

Install PyTorch, torchvision, CLIP dependencies, and clone the StyleGAN2-ADA-PyTorch repository. We use a fork with updated dependencies for compatibility with recent PyTorch versions. The CLIP model (ViT-B/32) is also installed from OpenAI's repository.

In [ ]:
!rm -rf stylegan2-ada-pytorch
!git clone https://github.com/RashmikaDushan/stylegan2-ada-pytorch.git

In [ ]:
%cd stylegan2-ada-pytorch
!git checkout dependency-updates

In [ ]:
!pip install git+https://github.com/openai/CLIP.git

## Step 2 – Mount Google Drive & Import Libraries

Mount Google Drive to access the pretrained StyleGAN2 checkpoint (`.pkl` file) and the image dataset stored there. Then import all required Python libraries including PyTorch, CLIP, and the StyleGAN2-ADA modules (`dnnlib`, `legacy`).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import sys
import copy
import math
import torch
import clip
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms, utils
from PIL import Image

In [ ]:
sys.path.append('./stylegan2-ada-pytorch')

import dnnlib
import legacy

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

## Step 3 – Load CLIP Model (ViT-B/32) and Set Device

Set the compute device to GPU (`cuda`) if available, otherwise CPU.

Load the **CLIP ViT-B/32** model which produces **512-dimensional** image embeddings. The CLIP model is **frozen** — its parameters are not updated during training. It serves two purposes:
1. **Encode real images** from the dataset into 512-dim embeddings (input to the mapping networks)
2. **Encode generated images** to compute the CLIP similarity loss against real image embeddings

In [ ]:
clip_model, _ = clip.load("ViT-B/32", device=device)
clip_model.eval()

for p in clip_model.parameters():
    p.requires_grad = False

## Step 4 – Configuration & Hyperparameters

Set the paths and training hyperparameters:
- **`CHECKPOINT_PATH`**: Path to the pretrained StyleGAN2 `.pkl` file on Google Drive (trained on FFHQ at 256×256)
- **`FFHQ_DIR`**: Path to the face image dataset used as real images
- **`BATCH_SIZE`**: Number of images per training step
- **`LR`**: Learning rate for Adam optimizer (only applied to the 3 mapping networks)
- **`LAMBDA_CLIP`**: Weight for the CLIP similarity loss term in the total loss: $\mathcal{L} = \mathcal{L}_{adv} + \lambda \cdot \mathcal{L}_{CLIP}$
- **`SAVE_DIR`**: Directory to save mapper checkpoints and sample images

In [ ]:
CHECKPOINT_PATH = '/content/drive/MyDrive/Research/stylegan2-ffhq-256x256.pkl'
FFHQ_DIR = '/content/drive/MyDrive/Research/sri_lankan_face_dataset'
BATCH_SIZE = 4
LR = 1e-4
EPOCHS = 100000
LAMBDA_CLIP = 1.0
SAVE_DIR = '/content/drive/MyDrive/Research/mapper3_checkpoints'

os.makedirs(FFHQ_DIR, exist_ok=True)
os.makedirs(SAVE_DIR, exist_ok=True)

## Step 5 – Load Pretrained StyleGAN2 (Generator & Discriminator)

Load the pretrained StyleGAN2 model from the `.pkl` checkpoint on Google Drive. This gives us:
- **`G` (Generator – G_ema)**: Contains both the **mapping network** and the **synthesis network**. We will deep-copy the mapping network 3 times to create our 3 mappers, and use the synthesis network (frozen) to generate images.
- **`D` (Discriminator)**: Used to compute the adversarial loss. Also **frozen** during training.

Both G and D are set to `eval()` mode and all their parameters have `requires_grad = False` — they are not updated during training. Only the 3 mapping networks we create will be trainable.

In [ ]:
print('Loading StyleGAN2 checkpoint (must be a .pkl from stylegan2-ada)...')

if not os.path.exists(CHECKPOINT_PATH):
    print(f'WARNING: checkpoint not found at {CHECKPOINT_PATH}. Upload the checkpoint to Colab or change the path.')
    G = None
    D = None
else:
    with open(CHECKPOINT_PATH, 'rb') as f:
        print('Opening checkpoint...')
        model_data = legacy.load_network_pkl(f)

    G = model_data['G_ema'].to(device)
    D = model_data['D'].to(device)

    print('Loaded G and D from checkpoint.')

In [ ]:
if G is not None:
    for p in G.parameters():
        p.requires_grad = False
    G.eval()

if D is not None:
    for p in D.parameters():
        p.requires_grad = False
    D.eval()

## Step 6 – CLIP Encoding Function

Define a function to encode image tensors through the frozen CLIP model:

1. Convert images from `[-1, 1]` range (StyleGAN2 output format) to `[0, 1]`
2. Resize to 224×224 (CLIP's expected input size)
3. Apply CLIP's ImageNet normalization
4. Pass through `clip_model.encode_image()` to get 512-dim embeddings
5. **Cast from float16 to float32** — CLIP on CUDA outputs half-precision, but our mapping networks operate in float32. This cast fixes the `RuntimeError: mat1 and mat2 must have the same dtype` error.
6. L2-normalize the embeddings

When `allow_grad=True`, gradients flow through the CLIP encoder (needed for the generated-image path so that the CLIP loss backpropagates to the mapping networks).

In [ ]:
clip_input_size = 224

def encode_with_clip_from_tensor(img_tensor, allow_grad=False):
    """
    img_tensor: (B, 3, H, W) in [-1, 1]
    Returns: (B, 512) float32 CLIP embedding, L2-normalized
    """
    x = (img_tensor + 1.0) / 2.0  # to [0, 1]
    x = F.interpolate(x, size=(clip_input_size, clip_input_size), mode='bilinear', align_corners=False)

    mean = torch.tensor([0.48145466, 0.4578275, 0.40821073], device=x.device).view(1, 3, 1, 1)
    std  = torch.tensor([0.26862954, 0.26130258, 0.27577711], device=x.device).view(1, 3, 1, 1)
    x = (x - mean) / std

    if allow_grad:
        emb = clip_model.encode_image(x)
    else:
        with torch.no_grad():
            emb = clip_model.encode_image(x)

    # CLIP on CUDA outputs float16 — cast to float32 to avoid dtype mismatch
    emb = emb.float()
    emb = emb / emb.norm(dim=-1, keepdim=True)
    return emb

## Step 7 – Image Preprocessing Transform

Define the transform pipeline for loading real images from the dataset:
1. Resize to 256×256 (matching StyleGAN2's output resolution)
2. Center crop to handle non-square images
3. Convert to tensor (`[0, 1]` range)
4. Normalize to `[-1, 1]` range (standard for StyleGAN2 and GAN training)

In [ ]:
real_image_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(256),
    transforms.ToTensor(),  # gives [0,1]
    transforms.Normalize([0.5,0.5,0.5],[0.5,0.5,0.5]) # -> [-1,1]
])

## Step 8 – Define the Three Mapping Networks (`ThreeMappers`)

This is the core trainable component. We create **3 mapping networks**, each initialised as a **deep copy of the pretrained StyleGAN2 mapping network** (`G.mapping`).

### How StyleGAN2's Mapping Network Works

In the original StyleGAN2, a single mapping network takes a 512-dim latent vector **z** and maps it to an intermediate latent vector **w** (also 512-dim). This **w** is then broadcast to all synthesis layers to form the **W** tensor of shape `(num_ws, 512)`. In our case, `num_ws = 14` for 256×256 resolution.

### Our Modification: 3 Separate Mappers

Instead of one mapping network, we use **3 independent mapping networks**. Each mapper:
1. **Input**: Takes the same 512-dim **CLIP embedding** of a real image
2. **Processing**: Passes it through the mapping network (8 fully-connected layers with LeakyReLU, as in the original StyleGAN2)
3. **Output**: Produces a 512-dim **w** vector
4. **Truncation**: Applies the truncation trick: $w' = \bar{w} + \psi \cdot (w - \bar{w})$ where $\bar{w}$ is the average latent from the pretrained model

### Layer Assignments

Each mapper's output **w** vector is assigned to specific synthesis layers:

```
Mapper 0 → w layers [0, 1, 2, 3]  → controls 4×4 and 8×8 blocks   (coarse structure)
Mapper 1 → w layers [4, 5, 6, 7]  → controls 16×16 and 32×32 blocks (medium features)
Mapper 2 → w layers [8–13]        → controls 64×64, 128×128, 256×256 (fine details)
```

The same **w** from each mapper is broadcast to all its assigned layers (e.g., Mapper 0's output fills layers 0, 1, 2, and 3). This gives us the complete **W+** tensor of shape `(B, 14, 512)` that the synthesis network expects.

### Comparison with 5-Mapper Version

| Aspect | 5-Mapper Version | 3-Mapper Version (this notebook) |
|--------|------------------|----------------------------------|
| # of mappers | 5 | 3 |
| Coarse control | 2 mappers (4×4, 8×8 separate) | 1 mapper (4×4 + 8×8 shared) |
| Medium control | 2 mappers (16×16, 32×32 separate) | 1 mapper (16×16 + 32×32 shared) |
| Fine control | 1 mapper (64×64–256×256) | 1 mapper (64×64–256×256) |
| Trainable params | ~5× mapping net | ~3× mapping net |

The 3-mapper version has fewer trainable parameters and couples coarse layers together, which may act as a regularizer and produce more coherent overall structure, at the cost of less independent control over individual resolution blocks.

### Why Deep Copy from Pretrained?

By starting from the pretrained mapping network weights, the mappers already understand the latent space W and can produce meaningful **w** vectors from the start. Training converges faster compared to randomly initialised networks.

In [ ]:
class ThreeMappers(nn.Module):
    """
    3 mapping networks, each deep-copied from the pretrained StyleGAN2 mapping network.
    Each takes a 512-dim CLIP embedding and outputs a 512-dim w vector.

    Layer assignments (for 256x256 resolution, num_ws=14):
        Mapper 0 → 4x4, 8x8           (w layers 0, 1, 2, 3)
        Mapper 1 → 16x16, 32x32       (w layers 4, 5, 6, 7)
        Mapper 2 → 64x64, 128x128, 256x256 (w layers 8–13)

    Truncation trick is applied: w' = w_avg + psi * (w - w_avg)
    """
    def __init__(self, pretrained_mapping, num_ws=14, w_dim=512, truncation_psi=0.7):
        super().__init__()
        self.num_ws = num_ws
        self.w_dim = w_dim
        self.truncation_psi = truncation_psi

        # Deep-copy pretrained mapping network 3 times
        self.mappers = nn.ModuleList([copy.deepcopy(pretrained_mapping) for _ in range(3)])

        # Enable gradient for all mapper parameters (they were frozen in the original G)
        for mapper in self.mappers:
            for p in mapper.parameters():
                p.requires_grad = True

        # Store w_avg from the pretrained model for truncation
        self.register_buffer('w_avg', pretrained_mapping.w_avg.clone())

        # Which synthesis layers each mapper feeds
        self.layer_assignments = [
            [0, 1, 2, 3],               # Mapper 0 → 4×4 + 8×8
            [4, 5, 6, 7],               # Mapper 1 → 16×16 + 32×32
            list(range(8, num_ws)),      # Mapper 2 → 64×64, 128×128, 256×256
        ]

    def forward(self, clip_embedding):
        """
        clip_embedding: (B, 512) float32 CLIP embedding
        Returns: ws (B, num_ws, w_dim) — the w+ tensor for the synthesis network
        """
        # Get one w vector per mapper
        w_vectors = []
        for mapper in self.mappers:
            # Use mapping network: z=clip_embedding, c=None (unconditional), no internal truncation
            w_broadcast = mapper(clip_embedding, c=None, truncation_psi=1.0)
            # w_broadcast: (B, num_ws, w_dim) — all layers identical
            w = w_broadcast[:, 0, :]  # (B, w_dim)

            # Truncation trick
            w = self.w_avg + self.truncation_psi * (w - self.w_avg)
            w_vectors.append(w)

        # Build w+ tensor: (B, num_ws, w_dim) by assigning each mapper's w to its target layers
        # Using torch.stack (no in-place ops) so autograd works cleanly
        layer_to_w = {}
        for mapper_idx, layers in enumerate(self.layer_assignments):
            for layer_idx in layers:
                layer_to_w[layer_idx] = w_vectors[mapper_idx]

        ws = torch.stack([layer_to_w[i] for i in range(self.num_ws)], dim=1)
        return ws

## Step 9 – Instantiate the Three Mappers

Read `num_ws` (14 for 256×256), `w_dim` (512), and `c_dim` (0 for unconditional FFHQ) from the loaded StyleGAN2 model. Then create the `ThreeMappers` module:
- Deep-copies `G.mapping` three times
- Enables gradients on all mapper parameters (since G was frozen)
- Stores `w_avg` from the pretrained model for truncation
- Sets truncation factor $\psi = 0.7$

In [ ]:
# Detect num_ws and w_dim from the pretrained model
num_ws = getattr(G.synthesis, 'num_ws', None) or getattr(G, 'num_ws', None) or 14
w_dim  = getattr(G, 'w_dim', 512)
c_dim  = getattr(G, 'c_dim', 0)

print(f'StyleGAN2 config: num_ws={num_ws}, w_dim={w_dim}, c_dim={c_dim}')

# Create 3 mapping networks, each initialised from the pretrained G.mapping
TRUNCATION_PSI = 0.7

mappers = ThreeMappers(
    pretrained_mapping=G.mapping,
    num_ws=num_ws,
    w_dim=w_dim,
    truncation_psi=TRUNCATION_PSI,
).to(device)

total_params     = sum(p.numel() for p in mappers.parameters())
trainable_params = sum(p.numel() for p in mappers.parameters() if p.requires_grad)
print(f'Three mapping networks created from pretrained StyleGAN2 mapping network.')
print(f'Total params: {total_params:,} | Trainable params: {trainable_params:,}')

## Step 10 – Dataset & DataLoader

Load the face image dataset from Google Drive. Each image is:
1. Resized and center-cropped to 256×256
2. Normalised to `[-1, 1]` range

The `DataLoader` serves batches of real images. These are fed through the frozen CLIP model to produce 512-dim embeddings, which are then passed to the 3 mapping networks. `num_workers=2` is used to avoid the worker process warning on Colab.

In [ ]:
class SimpleImageDataset(Dataset):
    def __init__(self, root, transform=None):
        self.root = root
        self.transform = transform
        self.image_files = sorted([
            os.path.join(root, f)
            for f in os.listdir(root)
            if f.lower().endswith(('.png', '.jpg', '.jpeg'))
        ])

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img = Image.open(self.image_files[idx]).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img

train_dataset = SimpleImageDataset(FFHQ_DIR, transform=real_image_transform)
train_loader  = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                           shuffle=True, num_workers=2, pin_memory=True)
print(f'Dataset loaded: {len(train_dataset)} images')

## Step 11 – Training Loop

The training pipeline for each batch:

### Forward Pass
1. **Real images** → Frozen **CLIP ViT-B/32** → 512-dim embedding `clip_real` (detached, no grad)
2. `clip_real` → **3 Mapping Networks** (trainable) → W+ tensor of shape `(B, 14, 512)`
3. W+ → Frozen **Synthesis Network** (`G.synthesis`) → Generated image `(B, 3, 256, 256)`

### Loss Computation
4. Generated image → Frozen **Discriminator** → Adversarial loss: $\mathcal{L}_{adv} = \text{softplus}(-D(x_{fake}))$
5. Generated image → Frozen **CLIP** → `clip_fake` embedding → CLIP loss: $\mathcal{L}_{CLIP} = 1 - \cos(\text{clip\_real}, \text{clip\_fake})$
6. **Total loss**: $\mathcal{L} = \mathcal{L}_{adv} + \lambda \cdot \mathcal{L}_{CLIP}$

### Backward Pass
7. Gradients flow **only** through the 3 mapping networks (all other components are frozen)
8. Adam optimizer updates the mapper parameters

### Checkpointing
- Every 500 steps: save mapper weights and a sample image grid to Google Drive

In [ ]:
optimizer = torch.optim.Adam(mappers.parameters(), lr=LR, betas=(0.9, 0.999))

step = 0
print_every = 50
save_every  = 500

print(f'Training config: lr={LR}, lambda_clip={LAMBDA_CLIP}, truncation_psi={TRUNCATION_PSI}')
print(f'Frozen: G.synthesis, D, clip_model  |  Trainable: mappers (3 mapping networks)')

for epoch in range(1, EPOCHS + 1):
    for batch_idx, real_imgs in enumerate(train_loader):
        mappers.train()
        optimizer.zero_grad()

        real_imgs = real_imgs.to(device)  # (B, 3, 256, 256) in [-1, 1]

        # ---- 1) CLIP-encode real images (frozen, no grad) ----
        clip_real = encode_with_clip_from_tensor(real_imgs, allow_grad=False).detach()
        # clip_real: (B, 512) float32

        # ---- 2) Map CLIP embeddings → w+ via the 3 mapping networks ----
        ws = mappers(clip_real)  # (B, num_ws, w_dim) float32

        # ---- 3) Synthesise images (frozen synthesis network) ----
        fake_images = G.synthesis(ws, noise_mode='const')
        # fake_images: (B, 3, 256, 256) in [-1, 1]

        # ---- 4) Discriminator loss (non-saturating G loss) ----
        try:
            d_fake = D(fake_images, c=torch.zeros(real_imgs.shape[0], c_dim, device=device))
        except TypeError:
            d_fake = D(fake_images)
        if isinstance(d_fake, (tuple, list)):
            d_fake = d_fake[0]
        g_adv_loss = F.softplus(-d_fake).mean()

        # ---- 5) CLIP similarity loss ----
        clip_fake = encode_with_clip_from_tensor(fake_images, allow_grad=True)
        # clip_fake: (B, 512) float32
        cos_sim   = F.cosine_similarity(clip_real, clip_fake, dim=-1)
        clip_diff = (1.0 - cos_sim).mean()

        # ---- 6) Total loss ----
        loss = g_adv_loss + LAMBDA_CLIP * clip_diff

        # ---- 7) Backward & update (only mapper params get gradients) ----
        loss.backward()
        optimizer.step()

        # ---- Logging ----
        if step % print_every == 0:
            print(f'[Step {step}] Epoch {epoch} Batch {batch_idx} | '
                  f'Loss: {loss.item():.4f} | Adv: {g_adv_loss.item():.4f} | '
                  f'CLIP diff: {clip_diff.item():.4f} | Cos sim: {cos_sim.mean().item():.4f}')

        # ---- Checkpointing ----
        if step % save_every == 0 and step > 0:
            ckpt_path = os.path.join(SAVE_DIR, f'mappers3_step_{step}.pt')
            torch.save({
                'mappers_state_dict': mappers.state_dict(),
                'optimizer':          optimizer.state_dict(),
                'step':               step,
                'epoch':              epoch,
                'truncation_psi':     TRUNCATION_PSI,
            }, ckpt_path)
            print(f'  → Saved checkpoint: {ckpt_path}')

            with torch.no_grad():
                sample_ws   = ws[:min(4, ws.shape[0])]
                sample_imgs = G.synthesis(sample_ws, noise_mode='const')
                grid = utils.make_grid((sample_imgs + 1) / 2, nrow=sample_imgs.shape[0])
                sample_path = os.path.join(SAVE_DIR, f'samples3_step_{step}.png')
                utils.save_image(grid, sample_path)
                print(f'  → Saved samples:    {sample_path}')

        step += 1

print('Training finished.')